In [1]:
#Verify L4 GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Thu Jul 16 19:21:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
#Verify PyTorch on L4
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Total GPU memory: 85.09 GB


In [3]:
#Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/msc_deepfake_project'
print(f"Project folder: {DRIVE_PROJECT}")

Mounted at /content/drive
Project folder: /content/drive/MyDrive/msc_deepfake_project


In [4]:
#Install HunyuanVideo dependencies
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece
!pip install -q einops safetensors

# Verify
import diffusers, transformers, torch
print(f"diffusers: {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print("\nAll HunyuanVideo dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 125.8 MB/s eta 0:00:00
diffusers: 0.39.0
transformers: 5.14.1
torch: 2.11.0+cu128

All HunyuanVideo dependencies installed.


In [9]:
#Loading HunyuanVideo on A100 (no CPU offload needed)
from diffusers import HunyuanVideoPipeline, HunyuanVideoTransformer3DModel
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("Loading HunyuanVideo model on A100...\n")

transformer = HunyuanVideoTransformer3DModel.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    subfolder="transformer",
    torch_dtype=torch.bfloat16
)

pipe = HunyuanVideoPipeline.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    transformer=transformer,
    torch_dtype=torch.bfloat16
)

# A100 has plenty of memory — move everything to GPU, no offload needed
pipe.to("cuda")

# Keep VAE optimisations for stability (they're safe on A100 too)
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("\nHunyuanVideo model loaded on A100 (no CPU offload).")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

Loading HunyuanVideo model on A100...



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


HunyuanVideo model loaded on A100 (no CPU offload).
GPU memory used: 63.71 GB
GPU memory available: 85.09 GB


In [10]:
#Generating on A100
import torch
from diffusers.utils import export_to_video
from datetime import datetime

prompt = "A woman with long brown hair smiling gently at the camera in a sunlit park, natural lighting, soft breeze moving her hair, cinematic quality"

generator = torch.Generator(device="cuda").manual_seed(42)

print(f"Prompt: {prompt}\n")
print("Generating video on A100...")

start_time = datetime.now()

video_frames = pipe(
    prompt=prompt,
    height=480,
    width=704,
    num_frames=73,
    num_inference_steps=30,
    guidance_scale=6.0,
    generator=generator,
).frames[0]

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\nGeneration complete in {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"Number of frames: {len(video_frames)}")

Prompt: A woman with long brown hair smiling gently at the camera in a sunlit park, natural lighting, soft breeze moving her hair, cinematic quality

Generating video on A100...


  0%|          | 0/30 [00:00<?, ?it/s]


Generation complete in 249.4 seconds (4.2 minutes)
Number of frames: 73


In [11]:
# Saving HunyuanVideo output to Drive
from datetime import datetime
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_id = f"hunyuan_001_{timestamp}"

video_path = f'{DRIVE_PROJECT}/videos/generated/hunyuan/{video_id}.mp4'
metadata_path = f'{DRIVE_PROJECT}/metadata/{video_id}.json'

# Export video
export_to_video(video_frames, video_path, fps=24)

# Save metadata (mirrors LTX metadata format for consistency)
metadata = {
    "video_id": video_id,
    "generator": "HunyuanVideo",
    "generator_version": "hunyuanvideo-community/HunyuanVideo",
    "prompt": prompt,
    "negative_prompt": None,
    "seed": 42,
    "width": 704,
    "height": 480,
    "num_frames": 73,
    "fps": 24,
    "num_inference_steps": 30,
    "guidance_scale": 6.0,
    "generation_time_seconds": duration,
    "generation_datetime": timestamp,
    "gpu_used": "L4",
    "optimizations": ["cpu_offload", "vae_tiling", "vae_slicing"]
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Video saved: {video_path}")
print(f"Metadata saved: {metadata_path}")

Video saved: /content/drive/MyDrive/msc_deepfake_project/videos/generated/hunyuan/hunyuan_001_20260716_193318.mp4
Metadata saved: /content/drive/MyDrive/msc_deepfake_project/metadata/hunyuan_001_20260716_193318.json


In [12]:
# Previewing the video inline
from IPython.display import Video
Video(video_path, embed=True, width=400)

In [14]:
#Downloading to local machine
from google.colab import files
files.download(video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>